# Notebook 02 — Cleaning & Preparation

**Goal:** apply every cleaning rule from the brief as its own function, keep a running log of dropped rows and the reason, and save clean tables for the join step.

**Input:** `data/interim/*_raw.parquet` **Output:** `data/interim/*_clean.parquet`, `data/rejects/rejected_validations.csv`

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 25)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REJECTS_DIR = PROJECT_ROOT / "data" / "rejects"
REJECTS_DIR.mkdir(parents=True, exist_ok=True)

validations = pd.read_parquet(INTERIM_DIR / "validations_raw.parquet")
routes = pd.read_parquet(INTERIM_DIR / "routes_raw.parquet")
vehicles = pd.read_parquet(INTERIM_DIR / "vehicles_raw.parquet")
stops = pd.read_parquet(INTERIM_DIR / "stops_raw.parquet")

RAW_VALIDATION_ROWS = len(validations)
print(f"Validations loaded: {RAW_VALIDATION_ROWS:,}")

Validations loaded: 81,600


## Drop log

Every time rows are removed, `log_drop` stores those rows with a `reject_reason` and adds their count to a running counter. At the end all rejected rows are written to `data/rejects/` and the counter is checked against the before/after row count.

In [3]:
rejected_frames: list[pd.DataFrame] = []
drop_counter: dict[str, int] = {}


def log_drop(rows: pd.DataFrame, reason: str) -> None:
    """Record dropped rows and their reason."""
    rows = rows.copy()
    rows["reject_reason"] = reason
    rejected_frames.append(rows)
    drop_counter[reason] = drop_counter.get(reason, 0) + len(rows)
    print(f"Dropped {len(rows):>5,} rows  -> {reason}")

# Part A — Validations

### A1. Duplicates

Exact copies are removed first, then any row whose `validation_id` was already seen (keeping the first occurrence). Doing it in this order means each dropped row is counted once, under the more specific reason.

In [4]:
def drop_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    exact = df.duplicated(keep="first")
    log_drop(df[exact], "Exact duplicate row")
    df = df[~exact]

    same_id = df.duplicated(subset="validation_id", keep="first")
    log_drop(df[same_id], "Duplicate validation_id (kept first)")
    return df[~same_id].reset_index(drop=True)


validations = drop_duplicates(validations)
assert validations["validation_id"].is_unique
print(f"Rows remaining: {len(validations):,}")

Dropped   802 rows  -> Exact duplicate row
Dropped   798 rows  -> Duplicate validation_id (kept first)
Rows remaining: 80,000


802 rows were exact copies and another 798 reused an existing ID with different values, so 1,600 rows were removed in total and every `validation_id` is now unique.

### A2. Strip whitespace from ID columns

In [5]:
ID_COLUMNS = ["validation_id", "route_id", "vehicle_id", "stop_id", "card_id"]


def strip_ids(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        df[col] = df[col].str.strip()
    return df


validations = strip_ids(validations, ID_COLUMNS)
print("Distinct route_id values after stripping:", validations["route_id"].nunique())

Distinct route_id values after stripping: 61


Before stripping there were 121 distinct `route_id` values; now there are 61 (the 60 real routes plus the invalid `R999`, handled in the integrity step).

### A3. Parse `validation_ts`

The three formats are parsed **one at a time with an explicit pattern**. A generic parser (`format="mixed"`) is not safe here: it reads the day/month value `06/03/2024` as 3 June instead of 6 March, which silently moves thousands of rows into the wrong month (and invents ridership in July–December, a period this data does not cover). Rows that match none of the patterns are rejected.

In [6]:
TS_FORMATS = {
    "iso": "%Y-%m-%d %H:%M:%S",      # 2024-03-15 07:42:11
    "eu": "%d/%m/%Y %H:%M",          # 15/03/2024 07:42
    "long": "%B %d, %Y %I:%M %p",    # March 15, 2024 7:42 AM
}


def parse_validation_ts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text = df["validation_ts"].str.strip()
    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    for name, fmt in TS_FORMATS.items():
        attempt = pd.to_datetime(text, format=fmt, errors="coerce")
        print(f"  {name:<5} format matched {attempt.notna().sum():>6,} rows")
        parsed = parsed.fillna(attempt)

    bad = parsed.isna()
    log_drop(df[bad], "Unparseable validation_ts")
    df["validation_ts"] = parsed
    return df[~bad].reset_index(drop=True)


validations = parse_validation_ts(validations)
print("Date range:", validations["validation_ts"].min(), "to", validations["validation_ts"].max())
print(validations["validation_ts"].dt.to_period("M").value_counts().sort_index().to_string())

  iso   format matched 56,212 rows
  eu    format matched 15,840 rows
  long  format matched  7,948 rows
Dropped     0 rows  -> Unparseable validation_ts
Date range: 2024-01-01 00:34:50 to 2024-06-30 23:48:50
validation_ts
2024-01    13688
2024-02    12734
2024-03    13593
2024-04    13147
2024-05    13786
2024-06    13052
Freq: M


All rows matched exactly one format, so nothing was rejected. The data covers **1 January – 30 June 2024**, with roughly 13,000 validations in each of the six months.

### A4. Normalise `fare_type`

Values are lower-cased and stripped, then mapped to the five allowed categories. The assertion stops the notebook if a new spelling ever appears, instead of letting it become a silent null.

In [7]:
FARE_TYPE_MAP = {
    "adult": "Adult", "full fare": "Adult",
    "student": "Student",
    "senior": "Senior", "senior citizen": "Senior",
    "child": "Child", "kid": "Child",
    "pass holder": "Pass Holder", "passholder": "Pass Holder", "monthly pass": "Pass Holder",
}
FARE_TYPES = ["Adult", "Student", "Senior", "Child", "Pass Holder"]


def normalise_fare_type(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    key = df["fare_type"].str.strip().str.lower()
    df["fare_type"] = key.map(FARE_TYPE_MAP)
    unmapped = key[df["fare_type"].isna()].unique()
    assert len(unmapped) == 0, f"Unmapped fare_type values: {list(unmapped)}"
    return df


validations = normalise_fare_type(validations)
validations["fare_type"].value_counts()

fare_type
Adult          44401
Student        11847
Senior          9471
Pass Holder     7958
Child           6323
Name: count, dtype: int64

Adult is the largest group (over half of all validations) once `Full Fare` is recognised as the same thing.

### A5. Normalise `transfer_flag`

In [8]:
BOOLEAN_MAP = {"Y": True, "1": True, "TRUE": True, "N": False, "0": False, "FALSE": False}


def to_boolean(series: pd.Series) -> pd.Series:
    key = series.astype("string").str.strip().str.upper()
    result = key.map(BOOLEAN_MAP)
    unmapped = key[result.isna() & key.notna()].unique()
    assert len(unmapped) == 0, f"Unmapped boolean values: {list(unmapped)}"
    return result.astype("boolean")


def normalise_transfer_flag(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["transfer_flag"] = to_boolean(df["transfer_flag"])
    return df


validations = normalise_transfer_flag(validations)
validations["transfer_flag"].value_counts(dropna=False)

transfer_flag
False    49573
True     30427
Name: count, dtype: Int64

All six spellings mapped and there are no nulls: about 38% of validations are transfers.

### A6. `passenger_count`

- **Null** → the row is dropped and logged: without a passenger count the boarding cannot be counted or priced.
- **Negative** → kept, and flagged with `is_reversed = True`. These are failed or reversed validations. Keeping the negative value means totals are *net* boardings (a reversal cancels the original boarding), which is what the agency actually carried.

In [9]:
def handle_passenger_count(df: pd.DataFrame) -> pd.DataFrame:
    missing = df["passenger_count"].isna()
    log_drop(df[missing], "Missing passenger_count")
    df = df[~missing].copy()
    df["passenger_count"] = df["passenger_count"].astype("int64")
    df["is_reversed"] = df["passenger_count"] < 0
    return df.reset_index(drop=True)


validations = handle_passenger_count(validations)
print("Reversed validations flagged:", validations["is_reversed"].sum())
validations["passenger_count"].value_counts().sort_index()

Dropped   811 rows  -> Missing passenger_count
Reversed validations flagged: 358


passenger_count
-3      126
-2      105
-1      127
 1    74066
 2     2355
 3     2410
Name: count, dtype: int64

### A7. `fare_amount`: impute nulls, cap outliers

1. **Nulls** are filled with the median fare **of the same fare type**. Fares differ a lot by type (Adult ≈ 2.50, Pass Holder ≈ 0), so one overall median would over-charge pass holders and under-charge adults. The median is used rather than the mean because the 25 extreme values (up to 9,999) would pull a mean far away from a normal fare.
2. **Outliers**: values above the 99.5th percentile are capped at that percentile, again **within each fare type**. A single global cap (≈ 2.86) would turn a 9,999 Pass Holder fare into 2.86 although pass holders normally pay 0; a per-type cap keeps each capped value realistic for that ticket. Capping instead of deleting keeps the boardings, which are valid even when the fare field is corrupt.

In [10]:
def handle_fare_amount(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    by_type = df.groupby("fare_type")["fare_amount"]

    was_null = df["fare_amount"].isna()
    df["fare_amount"] = df["fare_amount"].fillna(by_type.transform("median"))

    cap = df.groupby("fare_type")["fare_amount"].transform(lambda s: s.quantile(0.995)).round(2)
    was_capped = df["fare_amount"] > cap
    df["fare_amount"] = df["fare_amount"].clip(upper=cap)

    print(f"Imputed: {was_null.sum():,} nulls | Capped: {was_capped.sum():,} values")
    return df


validations = handle_fare_amount(validations)
validations.groupby("fare_type")["fare_amount"].describe()[["count", "min", "50%", "max"]]

Imputed: 1,156 nulls | Capped: 368 values


,count,min,50%,max
fare_type,,,,
Adult,43930.0,1.88,2.5,2.89
Child,6263.0,0.32,1.0,1.40
Pass Holder,7883.0,0.00,0.0,0.38
Senior,9376.0,0.60,1.2,1.60
Student,11737.0,0.95,1.5,1.88


After cleaning, the highest fare of each type is close to its normal price (Adult max ≈ 2.89) and no impossible values remain. By definition the cap also trims the top 0.5% of normal fares of each type, but only by a few cents, so the effect on revenue is negligible.

### A8. Missing `card_id` → `SINGLE_RIDE`

In [11]:
def fill_card_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    blank = df["card_id"].isna() | (df["card_id"] == "")
    df.loc[blank, "card_id"] = "SINGLE_RIDE"
    return df


validations = fill_card_id(validations)
print("SINGLE_RIDE validations:", (validations["card_id"] == "SINGLE_RIDE").sum())

SINGLE_RIDE validations: 9448


About 12% of boardings are single-ride paper tickets. They stay in the data; only the unique-card count in Notebook 04 leaves them out, because `SINGLE_RIDE` is not a real card.

# Part B — Master tables

### B1. Routes

In [12]:
def clean_routes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["route_id"] = df["route_id"].str.strip()
    df["route_name"] = df["route_name"].str.strip().str.title()
    df["mode"] = df["mode"].str.strip().str.title()
    df["district"] = df["district"].str.strip().fillna("Unknown")
    df["route_length_km"] = df["route_length_km"].fillna(
        df.groupby("mode")["route_length_km"].transform("median")
    )
    return df


routes = clean_routes(routes)
print(routes["mode"].value_counts().to_string())
print("Null route_length_km:", routes["route_length_km"].isna().sum())

mode
Bus      34
Tram     20
Metro     6
Null route_length_km: 0


Ten spellings collapse to three modes (34 bus, 20 tram and 6 metro routes). The 3 missing route lengths take the median length of their own mode.

### B2. Vehicles

`vehicle_age_band` is calculated from 2026, the upper limit of the valid year range in the brief.

In [13]:
REFERENCE_YEAR = 2026


def clean_vehicles(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["vehicle_id"] = df["vehicle_id"].str.strip()

    valid_year = df["year_manufactured"].between(1980, REFERENCE_YEAR)
    df["year_manufactured"] = df["year_manufactured"].where(valid_year)

    age = REFERENCE_YEAR - df["year_manufactured"]
    df["vehicle_age_band"] = (
        pd.cut(age, bins=[-1, 5, 10, 20, np.inf], labels=["0-5", "6-10", "11-20", "20+"])
        .astype("string")
        .fillna("Unknown")
    )

    df["capacity"] = df["capacity"].fillna(
        df.groupby("vehicle_type")["capacity"].transform("median").round()
    ).astype("int64")
    return df


vehicles = clean_vehicles(vehicles)
print("Invalid years set to null:", vehicles["year_manufactured"].isna().sum())
print(vehicles["vehicle_age_band"].value_counts().to_string())
vehicles.groupby("vehicle_type")["capacity"].agg(["min", "median", "max"])

Invalid years set to null: 15
vehicle_age_band
20+        84
11-20      44
6-10       23
Unknown    15
0-5        14


,min,median,max
vehicle_type,,,
Articulated Bus,104,118.0,137
Metro Set,514,619.0,690
Standard Bus,60,69.0,80
Tram,170,201.0,227


The 15 impossible years are now null and those vehicles fall into the `Unknown` age band. Almost half the fleet is more than 20 years old. Capacities are consistent per vehicle type, from about 70 places on a standard bus to about 600 on a metro set.

### B3. Stops

In [14]:
def clean_stops(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["stop_id"] = df["stop_id"].str.strip()
    df["stop_name"] = df["stop_name"].str.strip().str.title()
    df["has_shelter"] = to_boolean(df["has_shelter"])
    return df


stops = clean_stops(stops)
print(stops["has_shelter"].value_counts().to_string())
stops.head()

has_shelter
False    211
True     189


,stop_id,stop_name,district,zone,has_shelter
0,S0001,Matthew Radial Stop,Riverside,Zone 2,False
1,S0002,Brianna Avenue Stop,Centro,Zone 3,True
2,S0003,Jean Circle Stop,Northgate,Zone 2,False
3,S0004,Carlos Bypass Stop,Old Town,Zone 1,True
4,S0005,Jones Freeway Stop,Riverside,Zone 2,False


# Part C — Referential integrity

A validation is dropped if its route, vehicle or stop does not exist in the cleaned master tables. `card_id` is **not** part of this check, so `SINGLE_RIDE` validations are kept.

In [15]:
def check_referential_integrity(df: pd.DataFrame) -> pd.DataFrame:
    checks = {
        "route_id": routes["route_id"],
        "vehicle_id": vehicles["vehicle_id"],
        "stop_id": stops["stop_id"],
    }
    bad_any = pd.Series(False, index=df.index)
    for col, master in checks.items():
        bad = ~df[col].isin(master)
        print(f"{col:<11} not in master: {bad.sum():>3}  {sorted(df.loc[bad, col].unique())}")
        bad_any |= bad

    log_drop(df[bad_any], "Unknown route_id, vehicle_id or stop_id")
    return df[~bad_any].reset_index(drop=True)


validations = check_referential_integrity(validations)
print("SINGLE_RIDE validations kept:", (validations["card_id"] == "SINGLE_RIDE").sum())

route_id    not in master:  14  ['R999']
vehicle_id  not in master:  13  ['V9999']
stop_id     not in master:  13  ['S9999']
Dropped    40 rows  -> Unknown route_id, vehicle_id or stop_id
SINGLE_RIDE validations kept: 9441


Only the placeholder IDs `R999`, `V9999` and `S9999` fail the check (about 40 rows).

# Part D — Derived columns

In [16]:
TIME_PERIODS = ["Night (00-03)", "Early (04-06)", "AM Peak (07-09)",
                "Midday (10-15)", "PM Peak (16-18)", "Evening (19-23)"]


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ts = df["validation_ts"]
    df["date"] = ts.dt.normalize()
    df["year"] = ts.dt.year
    df["month"] = ts.dt.month
    df["year_month"] = ts.dt.strftime("%Y-%m")
    df["quarter"] = ts.dt.quarter
    df["hour"] = ts.dt.hour
    df["day_of_week"] = ts.dt.day_name()
    df["is_weekend"] = ts.dt.day_of_week >= 5
    df["time_period"] = pd.cut(
        df["hour"], bins=[-1, 3, 6, 9, 15, 18, 23], labels=TIME_PERIODS
    ).astype("string")
    df["revenue"] = (df["fare_amount"] * df["passenger_count"]).round(2)
    return df


validations = add_derived_columns(validations)
validations[["validation_ts", "date", "year_month", "quarter", "hour",
             "day_of_week", "is_weekend", "time_period", "revenue"]].head()

,validation_ts,date,year_month,quarter,hour,day_of_week,is_weekend,time_period,revenue
0,2024-05-13 18:54:29,2024-05-13,2024-05,2,18,Monday,False,PM Peak (16-18),2.53
1,2024-06-06 08:48:00,2024-06-06,2024-06,2,8,Thursday,False,AM Peak (07-09),2.48
2,2024-03-06 13:15:00,2024-03-06,2024-03,1,13,Wednesday,False,Midday (10-15),2.85
3,2024-04-21 17:30:00,2024-04-21,2024-04,2,17,Sunday,True,PM Peak (16-18),4.92
4,2024-06-14 18:35:27,2024-06-14,2024-06,2,18,Friday,False,PM Peak (16-18),1.34


In [17]:
validations["time_period"].value_counts().reindex(TIME_PERIODS)

time_period
Night (00-03)        700
Early (04-06)       2720
AM Peak (07-09)    23073
Midday (10-15)     19536
PM Peak (16-18)    25203
Evening (19-23)     7917
Name: count, dtype: int64[pyarrow]

The two peak periods (07–09 and 16–18) together hold more than half of all validations, although they cover only 6 hours of the day.

# Part E — Drop summary and save

In [18]:
summary = pd.DataFrame(
    {"rows": [RAW_VALIDATION_ROWS, *drop_counter.values(), len(validations)]},
    index=["Raw validations", *[f"  dropped: {r}" for r in drop_counter], "Clean validations"],
)
display(summary)

total_dropped = sum(drop_counter.values())
assert RAW_VALIDATION_ROWS - total_dropped == len(validations), "Drop log does not add up!"
print(f"Check passed: {RAW_VALIDATION_ROWS:,} - {total_dropped:,} = {len(validations):,}")

,rows
Raw validations,81600
dropped: Exact duplicate row,802
dropped: Duplicate validation_id (kept first),798
dropped: Unparseable validation_ts,0
dropped: Missing passenger_count,811
"dropped: Unknown route_id, vehicle_id or stop_id",40
Clean validations,79149


Check passed: 81,600 - 2,451 = 79,149


Every removed row is accounted for: the raw count minus the logged drops equals the clean count exactly.

In [19]:
rejects = pd.concat(rejected_frames, ignore_index=True)
rejects["validation_ts"] = rejects["validation_ts"].astype("string")
rejects.to_csv(REJECTS_DIR / "rejected_validations.csv", index=False, encoding="utf-8")
print(f"Saved {len(rejects):,} rejected rows")
print(rejects["reject_reason"].value_counts().to_string())

Saved 2,451 rejected rows
reject_reason
Missing passenger_count                    811
Exact duplicate row                        802
Duplicate validation_id (kept first)       798
Unknown route_id, vehicle_id or stop_id     40


In [20]:
outputs = {"validations": validations, "routes": routes, "vehicles": vehicles, "stops": stops}
for name, df in outputs.items():
    df.to_parquet(INTERIM_DIR / f"{name}_clean.parquet", index=False)
    print(f"Saved {name}_clean.parquet  {df.shape}")

Saved validations_clean.parquet  (79149, 21)
Saved routes_clean.parquet  (60, 6)
Saved vehicles_clean.parquet  (180, 6)
Saved stops_clean.parquet  (400, 5)


**Next:** `03_join_and_explore.ipynb` joins the four clean tables into one fact table.